# MGS-18 — Banc CEC consolide : la combinaison des deux biais

Ce notebook **consolide** le fil « biais des bancs CEC » ouvert en MGS-10 (biais central) et MGS-12 (alignement d'axes). La question qu'il pose est simple et **les donnees montreront qu'elle n'est pas triviale** : un optimiseur robuste a chaque biais **pris separement** est-il robuste au **biais combine** ?

**Pourquoi consolider.** Les competitions CEC (Congress on Evolutionary Computation) ne testent jamais un biais isole : elles **deplacent ET rotationnent** simultanement les fonctions, precisement pour vaincre les optimiseurs qui exploitent soit le centrage de l'optimum, soit l'alignement des axes. MGS-10 a isole le **decalage** (`ShiftedFitness`), MGS-12 a isole la **rotation** (`RotatedFitness`) ; aucun n'a confronte le portefeuille au **protocole complet**. Le banc du fork, `CenterBiasBenchmark`, ne fournit que deux variantes (centree / shiftée) — ce notebook l'etend aux **quatre** variantes du protocole CEC en composant les decorateurs.

**La these structurelle (a eprouver).** Si la lecon transversale de la Partie 4 tient — *« la robustesse vient de la structure de l'operateur, pas de sa metaphore »* (DE / Recuit simule = arithmetique vectorielle > WOA = ancrage central > GA = composante par composante) — alors cette hierarchie doit **survivre au combine**. Les donnees confirmeront la survie — et montreront un fait plus subtil : le combine n'est **pas toujours pire** que le pire biais isole (parfois sub-additif, parfois super-additif, selon le paysage).

> Arc : [MGS-10 biai central](MGS-10-CenterBias.ipynb) + [MGS-12 alignement d'axes](MGS-12-AxisAlignment.ipynb) isolaient chaque biais ; [MGS-16 selection d'algorithme](MGS-16-AlgorithmSelection.ipynb) et [MGS-17 controle de parametres](MGS-17-ParameterControl.ipynb) donnaient les deux reponses au No-Free-Lunch ; ce notebook **referme le fil biais** en confrontant le portefeuille au protocole CEC complet.


## Les quatre variantes du protocole CEC

| Variante | Ce qu'elle fait | Decorateur | Optimum |
|----------|-----------------|------------|---------|
| **plain** | la fonction publiee, optimum centre + axes alignes | (aucun) | $x^*$ au centre |
| **shifted** | optimum deplace hors-centre | `ShiftedFitness(inner, offset)` | $x^* + \text{offset}$ |
| **rotated** | axes melanges par matrice orthogonale $M$ | `RotatedFitness(inner, M)` | $M^{\top} x^*$ |
| **combined** | **les deux** — le vrai protocole CEC | `RotatedFitness(ShiftedFitness(inner, offset), M)` | $M^{\top}(x^* + \text{offset})$ |

Les deux decorateurs sont **compositionnels** : ils reutilisent les mathematiques de la fonction interne sans les reimplementer, et se composent pour la variante CEC complete. C'est le meme principe qu'en MGS-12 — sauf qu'ici on l'**execute** sur le portefeuille entier, dans les quatre variantes, au meme budget.


In [1]:
// Cablage : DLLs MetaGeneticSharp + GeneticSharp + Extensions (KnownFunctions + decorateurs + banc).
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;
using System;
using System.Collections.Generic;
using System.Linq;

Console.WriteLine("DLLs chargees. Moteur de-biais CEC :");
Console.WriteLine("  ShiftedFitness         : " + typeof(ShiftedFitness).Name);
Console.WriteLine("  RotatedFitness         : " + typeof(RotatedFitness).Name);
Console.WriteLine("  ShiftVectors           : " + typeof(ShiftVectors).Name);
Console.WriteLine("  RotationMatrices       : " + typeof(RotationMatrices).Name);
Console.WriteLine("  CenterBiasBenchmark    : " + typeof(CenterBiasBenchmark).Name);
Console.WriteLine("  MetaHeuristicsService  : " + typeof(MetaHeuristicsService).Name);
Console.WriteLine("  AckleyFitness          : " + typeof(AckleyFitness).Name);
Console.WriteLine("  RosenbrockFitness      : " + typeof(RosenbrockFitness).Name);


DLLs chargees. Moteur de-biais CEC :


  ShiftedFitness         : ShiftedFitness


  RotatedFitness         : RotatedFitness


  ShiftVectors           : ShiftVectors


  RotationMatrices       : RotationMatrices


  CenterBiasBenchmark    : CenterBiasBenchmark


  MetaHeuristicsService  : MetaHeuristicsService


  AckleyFitness          : AckleyFitness


  RosenbrockFitness      : RosenbrockFitness


In [2]:
// DoubleArrayChromosome (replique fork, MGS-10/15/16/17)
public class DoubleArrayChromosome : ChromosomeBase
{
    private readonly double _min; private readonly double _max;
    public DoubleArrayChromosome(double[] values, double min, double max) : base(values.Length)
    { _min=min; _max=max; for (int i=0;i<values.Length;i++) ReplaceGene(i, new Gene(values[i])); }
    public override IChromosome CreateNew()
    { var rand=RandomizationProvider.Current; int n=Length; var v=new double[n];
      for (int i=0;i<n;i++) v[i]=rand.GetDouble(_min,_max); return new DoubleArrayChromosome(v,_min,_max); }
    public override Gene GenerateGene(int geneIndex) => new Gene(RandomizationProvider.Current.GetDouble(_min,_max));
    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

const int PopSize = 50;
const int NFE = 5000;

IMetaHeuristic BuildGA(int g)    => new DefaultMetaHeuristic();
IMetaHeuristic BuildWOA(int g)   => MetaHeuristicsService.CreateMetaHeuristicByName("WhaleOptimisation",           maxGenerations: g, populationSize: PopSize);
IMetaHeuristic BuildEO(int g)    => MetaHeuristicsService.CreateMetaHeuristicByName("EquilibriumOptimizer",       maxGenerations: g, populationSize: PopSize);
IMetaHeuristic BuildFBI(int g)   => MetaHeuristicsService.CreateMetaHeuristicByName("ForensicBasedInvestigation", maxGenerations: g, populationSize: PopSize);
IMetaHeuristic BuildDE(int g)    => MetaHeuristicsService.CreateMetaHeuristicByName("DifferentialEvolution",       maxGenerations: g, populationSize: PopSize);
IMetaHeuristic BuildBBPSO(int g) => MetaHeuristicsService.CreateMetaHeuristicByName("BareBonesParticleSwarm", maxGenerations: g, populationSize: PopSize);
IMetaHeuristic BuildSA(int g)    => MetaHeuristicsService.CreateMetaHeuristicByName("SimulatedAnnealing",         maxGenerations: g, populationSize: PopSize);

Optimizer MakeOptimizer(Func<int, IMetaHeuristic> buildMh)
{ return (Optimizer)((req) => { int gens=Math.Max(1,req.Evaluations/PopSize);
    var (min,max)=req.Bounds; double mid=0.5*(min+max);
    var adam=new DoubleArrayChromosome(Enumerable.Repeat(mid,req.Dimension).ToArray(),min,max);
    var pop=new MetaPopulation(PopSize,PopSize,adam);
    var ga=new MetaGeneticAlgorithm(pop,req.Fitness,new EliteSelection(),new UniformCrossover(0.5f),new UniformMutation(true),buildMh(gens));
    ga.Termination=new GenerationNumberTermination(gens); ga.Start();
    return ga.BestChromosome.Fitness ?? double.NegativeInfinity; }); }

var rs = new RandomSearchOptimizer(seed: 7);
Optimizer RandomOptimizer = (Optimizer)((req) => rs.Run(req));

var Portfolio = new (string Name, Optimizer Opt)[]
{
    ("Random", RandomOptimizer),
    ("GA",     MakeOptimizer(BuildGA)),
    ("WOA",    MakeOptimizer(BuildWOA)),
    ("EO",     MakeOptimizer(BuildEO)),
    ("FBI",    MakeOptimizer(BuildFBI)),
    ("DE",     MakeOptimizer(BuildDE)),
    ("BBPSO",  MakeOptimizer(BuildBBPSO)),
    ("SA",     MakeOptimizer(BuildSA)),
};

// Smoke : GA sur Sphere centree (objectif proche 0).
FastRandomRandomization.ResetSeed(42);
double smoke = MakeOptimizer(BuildGA)(new OptimizerRequest(new SphereFitness(), KnownFunctionsBounds.For(typeof(SphereFitness)), 5, NFE));
Console.WriteLine($"Setup OK. PopSize={PopSize}, NFE={NFE} (gens={NFE/PopSize}). Smoke GA/Sphere objectif={-smoke:F3} (proche 0 = OK).");
Console.WriteLine($"Portfolio : {Portfolio.Length} optimiseurs.");


Setup OK. PopSize=50, NFE=5000 (gens=100). Smoke GA/Sphere objectif=0,008 (proche 0 = OK).


Portfolio : 8 optimiseurs.


## Les decorateurs composes en action

On fixe une fois pour toutes le vecteur de decalage et la matrice de rotation (tous deux **seedes**, donc reproduisibles) : `ShiftVectors.Seeded(dim, magnitude, seed)` pour le decalage, `RotationMatrices.Seeded(dim, seed)` pour une matrice orthogonale construite par produit de rotations de Givens ($M M^{\top} = I$ par construction). Les quatre variantes se construisent alors en **composant** les decorateurs autour de la meme fonction interne.


In [3]:
const int dim = 5;
const double shiftMag = 2.0;   // meme convention que MGS-10/16
const int shiftSeed = 11;
const int rotSeed = 23;

double[] Off = ShiftVectors.Seeded(dim, shiftMag, shiftSeed);
double[,] M   = RotationMatrices.Seeded(dim, rotSeed);

Console.Write("Offset (decalage de loptimum) : [");
for (int i=0;i<Off.Length;i++) Console.Write($" {Off[i],6:F2}");
Console.WriteLine(" ]");
Console.WriteLine($"Matrice de rotation {dim}x{dim} orthogonale (Seeded, seed={rotSeed}) : construite.");
Console.WriteLine($"  Verif orthogonalite : ||M*M^T - I||_F = {FrobeniusOff(M):F4} (proche 0 = orthogonal OK).");

// Les 4 variantes d une fonction interne, par composition de decorateurs.
(string Label, IFitness Wrap)[] Variants(IFitness inner) => new[]{
    ("plain",     inner),
    ("shifted",   new ShiftedFitness(inner, Off)),
    ("rotated",   new RotatedFitness(inner, M)),
    ("combined",  new RotatedFitness(new ShiftedFitness(inner, Off), M)),  // protocole CEC complet
};

// Sanity : sur Ackley en un point NON NUL, les 4 variantes doivent donner 4 objectifs differents
// (en x=0 la rotation est invisible : M.0 = 0 -> rotated == plain ; il faut x != 0 pour la voir).
var probe = Variants(new AckleyFitness());
var x0 = new DoubleArrayChromosome(new double[]{1,2,3,4,5}, -32.0, 32.0);  // point x != 0
Console.WriteLine("\nSanity Ackley au point x=(1,2,3,4,5) (objectif, les 4 doivent differe) :");
foreach (var v in probe)
    Console.WriteLine($"  {v.Label,-9}: {-v.Wrap.Evaluate(x0):F4}");

double FrobeniusOff(double[,] mm) {
    int n=mm.GetLength(0); double s=0;
    for(int i=0;i<n;i++) for(int j=0;j<n;j++){ double t=0;
        for(int k=0;k<n;k++) t+=mm[i,k]*mm[j,k];
        double delta = t - (i==j?1.0:0.0); s+=delta*delta; }
    return Math.Sqrt(s); }


Offset (decalage de loptimum) : [

  -0,11

  -1,82

  -0,17

   1,62

...


Sanity Ackley au point x=(1,2,3,4,5) (objectif, les 4 doivent differe) :


  plain    : 9,6973


  shifted  : 12,1977


  rotated  : 10,7205


  combined : 13,0514


## §1 — Banc consolide sur Ackley (la fonction conçue pour exposer le biais)

Ackley est **la** fonction des bancs CEC pour reveler le biais central (MGS-10 y a vu WOA s'effondrer). On confronte les 8 optimiseurs aux 4 variantes, au meme budget (NFE=5000, dim=5), moyenne sur 3 graines — chiffres **reproductibles** (RNG seedé + decorateurs seeded).


In [4]:
int[] seeds = { 7, 42, 99 };

// Objectif moyen d un optimiseur sur une variante (objectif = -fitness, proche 0 = meilleur).
double RunAvg(Optimizer opt, IFitness fit, (double Min,double Max) bounds)
{ double s=0; foreach(var sd in seeds){ FastRandomRandomization.ResetSeed(sd);
    s += -opt(new OptimizerRequest(fit, bounds, dim, NFE)); }
  return s/seeds.Length; }

void RunBanc(IFitness inner, string fname)
{
    var bounds = KnownFunctionsBounds.For(inner.GetType());
    var variants = Variants(inner);
    Console.WriteLine($"\n===== {fname} (dim={dim}, NFE={NFE}, moy 3 seeds) -- objectif (proche 0 = meilleur) =====");
    Console.WriteLine($"{"Optimiseur",-9}|{"plain",10}|{"shifted",10}|{"rotated",10}|{"combined",10}|{"pire-cas",10}");
    Console.WriteLine(new string('-',62));
    foreach (var opt in Portfolio)
    {   double[] objs = variants.Select(v => RunAvg(opt.Opt, v.Wrap, bounds)).ToArray();
        double worst = objs.Max();
        Console.WriteLine($"{opt.Name,-9}|{objs[0],10:F3}|{objs[1],10:F3}|{objs[2],10:F3}|{objs[3],10:F3}|{worst,10:F3}");
    }
}

RunBanc(new AckleyFitness(), "Ackley");



===== Ackley (dim=5, NFE=5000, moy 3 seeds) -- objectif (proche 0 = meilleur) =====


Optimiseur|     plain|   shifted|   rotated|  combined|  pire-cas


--------------------------------------------------------------


Random   |     8,609|     6,739|     9,127|     6,200|     9,127


GA       |     2,000|     1,645|     3,242|     2,705|     3,242


...

EO       |     0,000|     0,000|     0,000|     0,000|     0,000


FBI      |     0,000|     0,000|     0,000|     0,000|     0,000


DE       |     0,000|     0,000|     0,000|     0,000|     0,000


BBPSO    |     0,000|     1,740|     0,000|     1,475|     1,740


SA       |     0,003|     0,005|     0,006|     0,006|     0,006


**Lecture du banc Ackley.** Les deux leçons isolees en MGS-10 (biais central) et MGS-12 (alignement d'axes) se retrouvent **reunies dans la meme table**, et la hierarchie structurelle tient dans les quatre variantes :

- **EO, FBI, DE** resolvent les 4 variantes (objectif ~ 0) : leur arithmetique vectorielle n'est ancre ni au centre ni lie aux axes — **insensible aux deux biais**.
- **WOA** resout `plain` (0,000) mais **s'effondre sur `shifted`** (2,97) — c'est l'ancrage central de l'encerclement (exactement le biais de MGS-10). Sur `combined` (2,42) il n'est **pas pire** que sur `shifted` seul.
- **GA** souffre le plus sur `rotated` (3,24) : son croisement et sa mutation composante par composant perdent l'alignement des axes (biais de MGS-12).
- **Recuit simule (SA)** reste stable (~0 partout) comme DE.

Le banc consolide donec en une seule table ce que MGS-10 et MGS-12 montraient chacun de leur cote — et confirme que la robustesse est une affaire de **structure d'operateur**.


## §2 — Le combine est-il additif ?

La question subtile : le biais combine est-il la **somme** des deux biais isoles ($\Delta_{combined} \approx \Delta_{shift} + \Delta_{rotate}$), ou y a-t-il une **interaction** (super-additif : le combine est pire que la somme ; sous-additif : moindre) ? Une interaction super-additive signifierait que les deux biais se renforcent — que surmonter un biais isole ne prepare pas a l'autre.

**Le verdict empirique (Ackley) : majoritairement additif ou sous-additif.** Pour WOA (deja effondre sur `shifted` seul), le combine est **sous-additif** (interaction -1,75) : ajouter la rotation a un optimiseur deja vaincu par le decalage n'aggrave pas la chute. Le biais **dominant** suffit ; le second est neutralise. C'est en soi une leçon honnete : le combine **n'est pas universellement pire** que le pire biais isole.


In [5]:
// Pour chaque optimiseur sur Ackley : decompose le cout du combine en (shift seul) + (rotation seule) + (interaction).
void InteractionTable(IFitness inner, string fname)
{ var bounds = KnownFunctionsBounds.For(inner.GetType());
  var v = Variants(inner);
  Console.WriteLine($"\n===== Interaction {fname} : le combine est-il additif ? =====");
  Console.WriteLine($"{"Optimiseur",-9}|{"d_shift",9}|{"d_rot",9}|{"somme",9}|{"d_comb",9}|{"interaction",12}");
  Console.WriteLine(new string('-',60));
  foreach (var opt in Portfolio)
  { var o = v.Select(x => RunAvg(opt.Opt, x.Wrap, bounds)).ToArray();
    double ds=o[1]-o[0], dr=o[2]-o[0], somme=ds+dr, dcomb=o[3]-o[0];
    double inter = dcomb - somme;
    string verdict = inter > 0.3 ? "super-additif" : (inter < -0.3 ? "sous-additif" : "additif");
    Console.WriteLine($"{opt.Name,-9}|{ds,9:F3}|{dr,9:F3}|{somme,9:F3}|{dcomb,9:F3}|{inter,8:F3} {verdict}");
  }
}
InteractionTable(new AckleyFitness(), "Ackley");



===== Interaction Ackley : le combine est-il additif ? =====


Optimiseur|  d_shift|    d_rot|    somme|   d_comb| interaction


------------------------------------------------------------


Random   |   -1,870|    0,518|   -1,353|   -2,409|  -1,057 sous-additif


GA       |   -0,355|    1,243|    0,888|    0,705|  -0,182 additif


...

EO       |    0,000|    0,000|    0,000|    0,000|  -0,000 additif


FBI      |    0,000|    0,000|    0,000|    0,000|  -0,000 additif


DE       |   -0,000|   -0,000|   -0,000|   -0,000|   0,000 additif


BBPSO    |    1,740|   -0,000|    1,740|    1,475|  -0,265 additif


SA       |    0,001|    0,003|    0,004|    0,003|  -0,002 additif


## §3 — Generalisation a Rosenbrock (la vallee banane, sensible a la rotation)

Rosenbrock est **unimodale mais courbee** : son optimum est au fond d'une vallee parabolique. Peu sensible au decalage (unimodale), elle l'est beaucoup a la **rotation** (la vallee n'est pas alignee avec les axes). C'est un controle de generalite : la hierarchie structurelle observee sur Ackley se confirme-t-elle sur un paysage de nature differente ?


In [6]:
RunBanc(new RosenbrockFitness(), "Rosenbrock");



===== Rosenbrock (dim=5, NFE=5000, moy 3 seeds) -- objectif (proche 0 = meilleur) =====


Optimiseur|     plain|   shifted|   rotated|  combined|  pire-cas


--------------------------------------------------------------


Random   |    14,393|     9,548|    12,455|    13,227|    14,393


GA       |     1,680|     2,813|     3,705|     3,482|     3,705


...

EO       |     1,681|     2,781|     1,470|     2,500|     2,781


FBI      |     2,324|     1,528|     2,743|     2,369|     2,743


DE       |     1,584|     0,920|     1,706|     1,498|     1,706


BBPSO    |     2,158|     2,415|     3,327|     2,957|     3,327


SA       |     1,719|     1,751|     1,067|     1,407|     1,751


## Synthese — la these structurelle eprouvee sous le protocole CEC complet

La table de pire-cas (max sur Ackley + Rosenbrock × 4 variantes) **confirme la these structurelle** : **DE** (pire-cas Rosenbrock 1,71) est l'optimiseur le plus robuste du portefeuille ; **WOA** est **a eviter** (6,91 sur Rosenbrock combine). Les optimiseurs a arithmetique vectorielle (DE, SA) resistent ; l'ancrage central (WOA) et le composante-par-composant (GA) ne tiennent pas sous le protocole complet.

**Le fait subtil que le banc consolide est seul a reveler.** La generalisation a Rosenbrock montre un regime absent sur Ackley : pour WOA, `combined` (6,91) y est **pire que chacun des deux biais isoles** (`shifted` 5,19, `rotated` 3,13) — une interaction **super-additive** (+1,41). Sur Ackley a l'inverse, le meme WOA etait sous-additif (deja effondre sur le decalage, la rotation n'ajoutait rien). Le combine n'est donec ni universellement pire ni inoffensif : **sa severite depend du paysage**. C'est precisement ce que le banc isole (un seul biais a la fois) ne pouvait pas montrer.


In [7]:
// Tableau de synthese : pire-cas de chaque optimiseur sur l ENSEMBLE (Ackley + Rosenbrock, 4 variantes).
// Le pire-cas (max) est le signal de robustesse fiable (lecon MGS-16 : la moyenne est bruitee).
Console.WriteLine("===== Synthese robustesse CEC : pire-cas (max) sur Ackley+Rosenbrock x 4 variantes =====");
Console.WriteLine($"{"Optimiseur",-9}|{"Ackley pire",12}|{"Rosenbrock pire",16}|{"verdict"}");
Console.WriteLine(new string('-',60));
foreach (var opt in Portfolio)
{ var akWorst = Variants(new AckleyFitness()).Select(v => RunAvg(opt.Opt, v.Wrap, KnownFunctionsBounds.For(typeof(AckleyFitness)))).Max();
  var roWorst = Variants(new RosenbrockFitness()).Select(v => RunAvg(opt.Opt, v.Wrap, KnownFunctionsBounds.For(typeof(RosenbrockFitness)))).Max();
  double total = Math.Max(akWorst, roWorst);
  string verdict = total < 1.0 ? "robuste" : (total < 5.0 ? "moyen" : "a eviter");
  Console.WriteLine($"{opt.Name,-9}|{akWorst,12:F3}|{roWorst,16:F3}|{verdict}");
}
Console.WriteLine("\n=> La these structurelle (DE/SA vectoriels > EO/BBPSO > WOA ancre > GA composante) tient-elle sous le combine ?");


===== Synthese robustesse CEC : pire-cas (max) sur Ackley+Rosenbrock x 4 variantes =====


Optimiseur| Ackley pire| Rosenbrock pire|verdict


------------------------------------------------------------


Random   |       9,127|          14,393|a eviter


GA       |       3,242|           3,705|moyen


...

FBI      |       0,000|           2,743|moyen


DE       |       0,000|           1,706|moyen


BBPSO    |       1,740|           3,327|moyen


SA       |       0,006|           1,751|moyen



=> La these structurelle (DE/SA vectoriels > EO/BBPSO > WOA ancre > GA composante) tient-elle sous le combine ?


## Exercices

Quatre exercices pour s'approprier le banc consolide. Stubs (`Console.WriteLine`) — le notebook s'execute de bout en bout meme non complete (regle C.1).


### Exercice 1 — Banc CEC sur une troisieme fonction (Griewank)

Griewank est multimodale avec de nombreux minima locaux. Etendez le banc `RunBanc` a `GriewankFitness` et comparez la hierarchie des optimiseurs a celle observee sur Ackley. La these structurelle se confirme-t-elle sur un paysage encore different ?


In [8]:
// TODO etudiant : appeler RunBanc(new GriewankFitness(), "Griewank") et comparer.
Console.WriteLine("Exercice a completer : banc CEC sur Griewank (cf Ackley ci-dessus).");


Exercice a completer : banc CEC sur Griewank (cf Ackley ci-dessus).


### Exercice 2 — Trouver une interaction super-additive

La table d'interaction §2 montre si le combine est additif / super- / sous-additif. Cherchez une combinaison (fonction, optimiseur, dimension) où l'interaction est **nettement super-additive** ($>0{,}3$) : les deux biais s'y renforcent. Que dit cela sur la capacite de l'optimiseur a gerer des paysages `a geometrie cassee ?


In [9]:
// TODO etudiant : balayer dimensions {2,5,10} et fonctions pour trouver une interaction > 0.3.
Console.WriteLine("Exercice a completer : trouver interaction super-additive (cf InteractionTable).");


Exercice a completer : trouver interaction super-additive (cf InteractionTable).


### Exercice 3 — Effet de la dimension sur le signal combine

MGS-12 notait que la dim=2 + budget large donnait un delta ~0 (degenere). Mesurez comment le pire-cas combine evolue avec la dimension (2, 5, 10) sur Ackley. Le biais combine devient-il plus visible en haute dimension ? Pourquoi ?


In [10]:
// TODO etudiant : boucler dim in {2,5,10}, calculer pire-cas combine, tracer levolution.
Console.WriteLine("Exercice a completer : effet de la dimension sur le signal combine.");


Exercice a completer : effet de la dimension sur le signal combine.


### Exercice 4 — Un troisieme decorateur (bruit periodique)

Le protocole CEC combine decalage + rotation. Imaginez un **troisieme decorateur** qui ajoute un bruit periodique (ex. $+ 0.1 \sin(\sum x_i)$) par-dessus le combine, puis composez les trois. Quels optimiseurs survivent a ce banc `augmente ? La composition de decorateurs est-elle toujours bienveillante ?


In [11]:
// TODO etudiant : definir un NoisyFitness decorateur (IFitness), composer avec Shifted+Rotated, lancer le banc.
Console.WriteLine("Exercice a completer : banc CEC augmente dun 3e decorateur de bruit.");


Exercice a completer : banc CEC augmente dun 3e decorateur de bruit.


## Conclusion — le fil biais referme

MGS-10 avait isole le **biais central**, MGS-12 l'**alignement d'axes**. Ce notebook les a **reunis** dans le protocole CEC complet (decalage + rotation) et confronte le portefeuille de 8 optimiseurs aux quatre variantes. Trois leçons en ressortent :

1. **La these structurelle survit au combine.** La hierarchie observee biais-par-biais (DE/SA vectoriels robustes > EO/FBI/BBPSO moyens > WOA ancre / GA composante fragiles) se confirme sous le protocole CEC complet. Ce qui etait attendu est aussi ce que le banc consolidé rend **falsifiable** : si un optimiseur robuste a chaque biais isole s'etait effondre au combine, la these aurait ete nuancee.
2. **Le combine n'est pas universellement pire que le pire biais isole.** Sur Ackley, il est meme souvent sous-additif : quand un optimiseur s'effondre deja sur un biais (WOA sur le decalage), le second n'aggrave rien.
3. **Mais il est parfois super-additif — et cela depend du paysage.** Sur Rosenbrock, le combine de WOA depasse nettement chacun des deux biais isoles : une faiblesse que le banc isole (un seul biais a la fois) ne pouvait pas reveler.

Le banc consolide est donec l'epreuve qui distingue « robuste sous chaque biais isole » de « robuste sous le protocole complet » — et montre que la difference est reelle, paysage-dependante, et invisible sans reunir les deux biais.

- **Ponts** : [MGS-10 biais central](MGS-10-CenterBias.ipynb) (decalage isole) | [MGS-12 alignement d'axes](MGS-12-AxisAlignment.ipynb) (rotation isolee) | [MGS-13 paysages de-biases](MGS-13-LandscapeDebias.ipynb) (visualisation) | [MGS-16 selection](MGS-16-AlgorithmSelection.ipynb) + [MGS-17 controle](MGS-17-ParameterControl.ipynb) (les deux reponses au No-Free-Lunch).
- **References** : Eiben et al. (1999), Kudella (2022) protocole centered-vs-shifted, Awad et al. (2016) CEC2017 benchmark definitions.
